In [18]:
import pandas as pd
import numpy as np
import cufflinks as cf
from plotly.offline import init_notebook_mode, plot, iplot
import plotly.express as px
from IPython.display import display

In [9]:
df = pd.read_csv('billionaires.csv')

In [10]:
df.head()

,Name,NetWorth,Country,Source,Rank,Age,Residence,Citizenship,Status,Children,Education,Self_made,geometry
0,Jeff Bezos,177.0,United States,Amazon,1,57.0,"Seattle, Washington",United States,In Relationship,4.0,"Bachelor of Arts/Science, Princeton University",True,POINT (-122.3300624 47.6038321)
1,Elon Musk,151.0,United States,"Tesla, SpaceX",2,49.0,"Austin, Texas",United States,In Relationship,7.0,"Bachelor of Arts/Science, University of Pennsy...",True,POINT (-97.74369950000001 30.2711286)
2,Bernard Arnault & family,150.0,France,LVMH,3,72.0,"Paris, France",France,Married,5.0,"Bachelor of Arts/Science, Ecole Polytechnique ...",False,POINT (2.3514616 48.8566969)
3,Bill Gates,124.0,United States,Microsoft,4,65.0,"Medina, Washington",United States,Divorced,3.0,"Drop Out, Harvard University",True,POINT (-122.2264453 47.620548)
4,Mark Zuckerberg,97.0,United States,Facebook,5,36.0,"Palo Alto, California",United States,Married,2.0,"Drop Out, Harvard University",True,POINT (-122.1598465 37.4443293)


In [11]:
init_notebook_mode(connected=True)
cf.go_offline()

In [12]:
df['Self_made'].replace(['True','False'], ['Self-made','Not self-made'], inplace=True)
df.columns = [col_n.lower() for col_n in df.columns]

In [20]:
n_duplicates = df.duplicated().sum()
df_described = df.describe().round(3)
null_cnts = df.isnull().sum()
null_pcts = (df.isnull().sum() / len(df)).round(3)
df_null = pd.DataFrame({'n_null': null_cnts, 
              'pct_null': null_pcts}).sort_values('n_null', ascending=False)

print(f"Dataframe Shape: {df.shape}")
print(f"Duplicate Rows: {n_duplicates}\n")
print(f"Numerical Column Description:")
display(df_described)
print(f"All Column Null Summary:")
display(df_null)

Dataframe Shape: (2755, 13)
Duplicate Rows: 0

Numerical Column Description:


,networth,rank,age,children
count,2755.000,2755.000,2630.000,1552.000
mean,4.749,1345.664,63.267,2.978
std,9.615,772.670,13.479,1.619
min,1.000,1.000,18.000,1.000
25%,1.500,680.000,54.000,2.000
50%,2.300,1362.000,63.000,3.000
75%,4.200,2035.000,73.000,4.000
max,177.000,2674.000,99.000,23.000


All Column Null Summary:


,n_null,pct_null
education,1346,0.489
children,1203,0.437
status,665,0.241
age,125,0.045
residence,40,0.015
self_made,18,0.007
citizenship,16,0.006
name,0,0.000
networth,0,0.000
country,0,0.000


In [21]:
df.fillna('NULL', inplace=True)

In [24]:
top_20_worth = df.sort_values('networth', ascending=False).iloc[:20] 
top_20_worth_fig = top_20_worth.figure(kind="bar", 
                   x="name", 
                   y="networth", 
                   title="Net Worth and Wealth Source of Top 20 Wealthiest Billionaires", 
                   xTitle="name", 
                   yTitle="Net Worth (Billions $USD)",
                   color="blue",
                   text="source")
top_20_worth_fig.update_yaxes(nticks=10)
display(top_20_worth_fig)

In [25]:
# Distribution of wealth

networthhist01 = df[['networth']].figure(kind="histogram",
                bins = (0, 200, 5),
                histnorm = 'percent',
                title = 'Histogram of networth of all individuals',
                xTitle = 'Net Worth (Billions in USD)',
                yTitle = 'Percentage',
                theme = 'ggplot',
                bargap = 0.1,
                color = 'blue',
                orientation = 'v',
                text = 'networth')

networthhist01.update_yaxes(nticks=20)
networthhist01.update_xaxes(nticks=20)

display(networthhist01)

In [28]:
# Distribution of wealth

networthhist02 = df[['networth']].figure(kind="histogram",
                bins = (0, 10, 1),
                histnorm = 'percent',
                title = 'Histogram of networth of all individuals',
                xTitle = 'Net Worth (Billions in USD)',
                yTitle = 'Frequency',
                theme = 'ggplot',
                bargap = 0.1,
                color = 'blue',
                orientation = 'v',
                text = 'networth')

networthhist02.update_yaxes(nticks=20)
networthhist02.update_xaxes(nticks=10)

display(networthhist02)

In [34]:
def plot_worth_unstacked(col):
    """
    Makes the following three plots relating net worth to a group (i.e. categorical column):
        1. Bar chart of total net worth by group
        2. Bar chart of number of billionaires by group
        3. Bar chart of average net worth per billionaire in each group
    Returns nothing but displays all three plots
    
    Parameters:
        col (str): the column to group by and analyze
    """
    sdf = df.copy()
    if col == 'children' or col == 'age':
        sdf[col].replace('NULL', -1, inplace=True)
    sdf = sdf.groupby(col).agg(
        {'networth': 'sum', 'name': 'count'}).sort_values('networth', ascending=False).round(3)
    #We only plot the top 20 values by total net worth
    if len(sdf.index) > 20:
        sdf = sdf.iloc[0:20]
    sdf['avg_networth'] = (sdf['networth'] / sdf['name']).round(3)
    n = len(sdf.index.unique())

    #Total net worth per category
    total_worth_fig = px.bar(sdf,
                 y="networth", 
                 text="networth", 
                 title=f"Total Net Worth by {col.capitalize()} (Top {n})")
    total_worth_fig.update_traces(textposition="outside", marker_color="lightsalmon")
    total_worth_fig.update_layout(yaxis={'title': 'Total Net Worth'}, xaxis={'nticks':30})

    #Number of billionaires per category
    count_fig = px.bar(sdf, y="name", text="name", 
                       title=f"Number of Billionaires by {col.capitalize()} (Top {n} by Total Net Worth)")
    count_fig.update_traces(textposition="outside", 
                            marker_color="crimson")
    count_fig.update_layout(yaxis={'title': 'Number of Billionaires'}, xaxis={'nticks':30})

    #Worth per billionaire per category
    avg_worth_fig = px.bar(sdf, 
                 y="avg_networth", 
                 text="avg_networth", 
                 title=f"Average Net Worth by {col.capitalize()} (Top {n} by Total Net Worth)")
    avg_worth_fig.update_traces(textposition="outside", marker_color="lightslategray")
    avg_worth_fig.update_layout(yaxis={'title': 'Average Net Worth per Person'}, xaxis={'nticks':30})
    
    display(total_worth_fig, count_fig, avg_worth_fig)
    return None

In [35]:
#Remove null safe_made values
df_selfmade = df[~df['self_made'].isnull()]

#Make histogram of net worth faceted by self made status
selfmade_hist = px.histogram(df_selfmade, 
                   x="networth", 
                   facet_row="self_made", 
                   range_x=(0, 50), 
                   range_y=(0, 1600),
                   facet_col_spacing=0.05,
                   nbins=40, 
                   title="Histograms of Net Worth by Self-made Status")

selfmade_hist.update_layout(bargap=0.1, yaxis={'range':[0,1800]}, xaxis={'nticks':20})
#Make labels for each subplot (right hand side) nicer
selfmade_hist.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

#Plot total worth by self made status
plot_worth_unstacked("self_made")


In [36]:
display(selfmade_hist)

In [37]:
# How does wealthy vary by country?
plot_worth_unstacked('country')

In [38]:
plot_worth_unstacked('source')

In [39]:
plot_worth_unstacked('children')

In [40]:
plot_worth_unstacked('status')

In [41]:
import ipywidgets as widgets
from ipywidgets import interact, interactive

In [42]:
cols_to_analyze = ['self_made', 'country','source','children','status']

full_description_style = {'description_width': 'initial'}
col_widget = widgets.Dropdown(description='Column to analyze:',
            options=cols_to_analyze,
            style=full_description_style)

interactive_output = interactive(plot_worth_unstacked, col=col_widget)
display(interactive_output)

interactive(children=(Dropdown(description='Column to analyze:', options=('self_made', 'country', 'source', 'c…